# 🧪 Quick Test — Model A & B

Test nhanh Model A (SimpleCNN) và Model B (ResNet101) trên validation set.

In [1]:
import os, sys, json, torch, random
import numpy as np
from PIL import Image
from torchvision import transforms
import matplotlib.pyplot as plt
%matplotlib inline

PROJECT_ROOT = "/home/mayxin/workspace/DeepLearning/new_vqa"
os.chdir(PROJECT_ROOT)
sys.path.insert(0, 'src')

from vocab import Vocabulary
from inference import get_model, greedy_decode, beam_search_decode, strip_compiled_prefix

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

# ── Load vocab ───────────────────────────────────────────────────────────────
vocab_q = Vocabulary(); vocab_q.load('data/processed/vocab_questions.json')
vocab_a = Vocabulary(); vocab_a.load('data/processed/vocab_answers.json')
print(f"Q-vocab: {len(vocab_q)} | A-vocab: {len(vocab_a)}")

# ── Load val data ────────────────────────────────────────────────────────────
val_data = json.load(open('data/vqa_e/VQA-E_val_set.json'))
print(f"Val samples: {len(val_data):,}")

# ── Image transform ──────────────────────────────────────────────────────────
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# ── Load Model A & B ─────────────────────────────────────────────────────────
models = {}
for m in ('A', 'B'):
    ckpt = f'checkpoints/model_{m.lower()}_best.pth'
    if os.path.exists(ckpt):
        model = get_model(m, len(vocab_q), len(vocab_a), glove_dim=300)
        sd = torch.load(ckpt, map_location='cpu', weights_only=True)
        model.load_state_dict(strip_compiled_prefix(sd))
        model.to(DEVICE).eval()
        models[m] = model
        print(f"✅ Model {m} loaded from {ckpt}")
    else:
        print(f"❌ Model {m}: {ckpt} not found")

print(f"\n{len(models)} models ready: {list(models.keys())}")

Device: cuda
Q-vocab: 4546 | A-vocab: 8648
Val samples: 88,488


OutOfMemoryError: CUDA out of memory. Tried to allocate 18.00 MiB. GPU 0 has a total capacity of 11.63 GiB of which 26.75 MiB is free. Process 339071 has 9.77 GiB memory in use. Including non-PyTorch memory, this process has 126.00 MiB memory in use. Of the allocated memory 5.93 MiB is allocated by PyTorch, and 16.07 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## Random Val Samples — Greedy Decode

Chọn ngẫu nhiên mẫu từ val set, hiển thị ảnh + Q + GT + prediction A/B.

In [ ]:
# ── Test trên random samples ─────────────────────────────────────────────────
N = 8  # số mẫu muốn test
random.seed(None)  # random thật sự mỗi lần chạy
samples = random.sample(val_data, N)

for i, ann in enumerate(samples):
    img_id = ann['img_id']
    q_text = ann['question']
    gt_ans = ann.get('multiple_choice_answer', '')
    gt_exp = ann.get('explanation', [''])[0] if ann.get('explanation') else ''
    gt_full = f"{gt_ans} because {gt_exp}" if gt_exp else gt_ans

    img_path = f"data/raw/val2014/COCO_val2014_{img_id:012d}.jpg"
    if not os.path.exists(img_path):
        print(f"⚠ Image not found: {img_path}")
        continue

    img_pil = Image.open(img_path).convert("RGB")
    img_tensor = transform(img_pil)
    q_tensor = torch.tensor(vocab_q.numericalize(q_text), dtype=torch.long)

    # Show image
    fig, ax = plt.subplots(1, 1, figsize=(4, 3))
    ax.imshow(img_pil)
    ax.axis('off')
    ax.set_title(f"[{i+1}] Q: {q_text}", fontsize=9, wrap=True)
    plt.tight_layout()
    plt.show()

    # Predictions
    print(f"  GT : {gt_full}")
    for m, model in models.items():
        with torch.no_grad():
            pred = greedy_decode(model, img_tensor, q_tensor, vocab_a, device=DEVICE)
        print(f"  {m}  : {pred}")
    print("─" * 60)

## Beam Search Test

So sánh greedy vs beam search (width=5, no_repeat_ngram=3).

In [ ]:
# ── Greedy vs Beam Search ─────────────────────────────────────────────────────
test_samples = random.sample(val_data, 5)

for i, ann in enumerate(test_samples):
    img_id = ann['img_id']
    q_text = ann['question']
    gt_ans = ann.get('multiple_choice_answer', '')
    gt_exp = ann.get('explanation', [''])[0] if ann.get('explanation') else ''
    gt_full = f"{gt_ans} because {gt_exp}" if gt_exp else gt_ans

    img_path = f"data/raw/val2014/COCO_val2014_{img_id:012d}.jpg"
    if not os.path.exists(img_path):
        continue

    img_pil = Image.open(img_path).convert("RGB")
    img_tensor = transform(img_pil)
    q_tensor = torch.tensor(vocab_q.numericalize(q_text), dtype=torch.long)

    print(f"[{i+1}] Q: {q_text}")
    print(f"     GT: {gt_full}")
    for m, model in models.items():
        with torch.no_grad():
            greedy = greedy_decode(model, img_tensor, q_tensor, vocab_a, device=DEVICE)
            beam = beam_search_decode(model, img_tensor, q_tensor, vocab_a,
                                      device=DEVICE, beam_width=5, no_repeat_ngram_size=3)
        print(f"     {m} greedy: {greedy}")
        print(f"     {m} beam-5: {beam}")
    print("─" * 70)

## Quick BLEU/METEOR — 500 samples

In [ ]:
# ── Quick Metrics trên 500 samples ────────────────────────────────────────────
!python src/compare.py --epoch 30 --num_samples 500

## Tự nhập câu hỏi

Chọn random ảnh, tự gõ câu hỏi để test.

In [ ]:
# ── Random ảnh + tự nhập câu hỏi ─────────────────────────────────────────────
# Đổi question bên dưới rồi chạy lại cell này

question = "What color is the sky?"  # ← ĐỔI CÂU HỎI Ở ĐÂY

# Random 1 ảnh từ val set
ann = random.choice(val_data)
img_id = ann['img_id']
img_path = f"data/raw/val2014/COCO_val2014_{img_id:012d}.jpg"

img_pil = Image.open(img_path).convert("RGB")
img_tensor = transform(img_pil)
q_tensor = torch.tensor(vocab_q.numericalize(question), dtype=torch.long)

fig, ax = plt.subplots(1, 1, figsize=(5, 4))
ax.imshow(img_pil)
ax.axis('off')
ax.set_title(f"Q: {question}", fontsize=10)
plt.tight_layout()
plt.show()

for m, model in models.items():
    with torch.no_grad():
        pred = greedy_decode(model, img_tensor, q_tensor, vocab_a, device=DEVICE)
    print(f"  Model {m}: {pred}")

## Grid View — 2×2 samples

In [ ]:
# ── 2×2 Grid — ảnh + predictions ──────────────────────────────────────────────
grid_samples = random.sample(val_data, 4)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for ax, ann in zip(axes.flatten(), grid_samples):
    img_id = ann['img_id']
    q_text = ann['question']
    gt_ans = ann.get('multiple_choice_answer', '')

    img_path = f"data/raw/val2014/COCO_val2014_{img_id:012d}.jpg"
    if not os.path.exists(img_path):
        ax.text(0.5, 0.5, 'Image not found', ha='center', va='center')
        ax.axis('off')
        continue

    img_pil = Image.open(img_path).convert("RGB")
    img_tensor = transform(img_pil)
    q_tensor = torch.tensor(vocab_q.numericalize(q_text), dtype=torch.long)

    preds = {}
    for m, model in models.items():
        with torch.no_grad():
            preds[m] = greedy_decode(model, img_tensor, q_tensor, vocab_a, device=DEVICE)

    ax.imshow(img_pil)
    ax.axis('off')

    pred_lines = '\n'.join(f"{m}: {p[:60]}" for m, p in preds.items())
    title = f"Q: {q_text}\nGT: {gt_ans}\n{pred_lines}"
    ax.set_title(title, fontsize=7, wrap=True, loc='left')

plt.suptitle("Model A vs B — Random Val Samples", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()